# Guidelines Dataset Explorer
Browse the filtered `epfl-llm/guidelines` subsets for each diagnostic test.

In [ ]:
import os
import pandas as pd
from datasets import load_from_disk
from IPython.display import display, HTML

pd.set_option('display.max_colwidth', 300)

# Works whether Jupyter is launched from repo root or from knowledge-graphs/
_cwd = os.getcwd()
_candidates = [
    os.path.join(_cwd, 'knowledge-graphs', 'data', 'guidelines_filtered'),  # from repo root
    os.path.join(_cwd, 'data', 'guidelines_filtered'),                       # from knowledge-graphs/
]
BASE = next((p for p in _candidates if os.path.exists(p)), _candidates[0])

TESTS = ['ecg', 'testicular_ultrasound', 'arm_xray', 'appendix_ultrasound', 'abdominal_ultrasound']
print('Base path:', BASE)

## Overview — row counts per test

In [2]:
dfs = {}
for test in TESTS:
    path = os.path.join(BASE, test)
    ds = load_from_disk(path)
    df = ds.to_pandas()
    dfs[test] = df

summary = pd.DataFrame([
    {'test': t, 'rows': len(df), 'sources': ', '.join(sorted(df['source'].unique()))}
    for t, df in dfs.items()
])
display(summary)

FileNotFoundError: Directory knowledge-graphs/data/guidelines_filtered/ecg not found

## Select a test to browse

In [ ]:
# Change this to any of: ecg | testicular_ultrasound | arm_xray | appendix_ultrasound | abdominal_ultrasound
ACTIVE_TEST = 'ecg'

### Row counts by source

In [ ]:
df = dfs[ACTIVE_TEST]
display(df['source'].value_counts().rename_axis('source').reset_index(name='count'))

### Titled rows (title is not null)

In [ ]:
titled = df[df['title'].notna() & (df['title'] != 'None')][['source', 'title', 'overview', 'url']].reset_index(drop=True)
print(f'{len(titled)} rows with titles')
display(titled.head(5))

### Keyword search in clean_text

In [ ]:
SEARCH_TERM = 'chest pain'   # <-- change this

hits = df[df['clean_text'].str.contains(SEARCH_TERM, case=False, na=False)].reset_index(drop=True)
print(f'{len(hits)} rows matching "{SEARCH_TERM}"')
display(hits[['source', 'title', 'overview']].head(5))

### Sentences containing ECG mentions

In [ ]:
import re

ECG_PATTERN = re.compile(r'\b(ecg|ekg|electrocardiogr\w*)\b', re.IGNORECASE)

records = []
for _, row in dfs['ecg'].iterrows():
    text = row['clean_text'] or ''
    for sent in re.split(r'(?<=[.!?])\s+', text):
        if ECG_PATTERN.search(sent):
            records.append({
                'source': row['source'],
                'title': row['title'] if row['title'] != 'None' else None,
                'sentence': sent.strip(),
            })

ecg_sentences = pd.DataFrame(records).reset_index(drop=True)
print(f'{len(ecg_sentences)} sentences mention ECG/EKG/electrocardiogram')
display(ecg_sentences.head(5))

### View full clean_text for a selected ECG sentence

In [ ]:
SENTENCE_IDX = 4   # <-- change to any index from ecg_sentences above

sel = ecg_sentences.iloc[SENTENCE_IDX]
# Look up the source row in dfs['ecg'] by matching source + title
mask = (dfs['ecg']['source'] == sel['source'])
if sel['title']:
    mask &= (dfs['ecg']['title'] == sel['title'])
matches = dfs['ecg'][mask]

print(f"Sentence [{SENTENCE_IDX}]: {sel['sentence']}")
print(f"Source  : {sel['source']}")
print(f"Title   : {sel['title']}")
print(f"\n{'='*80}\n")

if matches.empty:
    print("No matching row found in dfs['ecg'].")
else:
    for i, (_, row) in enumerate(matches.iterrows()):
        if len(matches) > 1:
            print(f"--- Match {i+1} of {len(matches)} ---")
        print(row['clean_text'])
        print()

### Read a full guideline

In [ ]:
ROW_INDEX = 0   # <-- change to any index from the search results above

row = df.iloc[ROW_INDEX]
print(f"Source : {row['source']}")
print(f"Title  : {row['title']}")
print(f"URL    : {row['url']}")
print(f"Overview: {row['overview']}")
print('\n' + '='*80 + '\n')
print(row['clean_text'])

## Combined dataset (all tests, deduped)

In [ ]:
combined = load_from_disk(os.path.join(BASE, 'combined')).to_pandas()
print(f'Combined: {len(combined)} unique rows')
display(combined[['source', 'diagnostic_tests', 'title', 'overview']].head(5))

### Rows matching multiple tests

In [ ]:
multi = combined[combined['diagnostic_tests'].str.contains(',')].reset_index(drop=True)
print(f'{len(multi)} rows match more than one test')
display(multi[['source', 'title', 'diagnostic_tests']].head(5))